[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/josephson/josephson.ipynb)

# Fluxons in a Josephson Junction

This example computes fluxons in a planar Josephson junction: lines of magnetic flux, one flux quantum per unit of their length, that travel through the thin barrier between two superconductors as relativistic strings. They contract as their speed nears the junction's own light speed, a closed loop of them collapses under its own tension, and one that runs into the junction's open edge comes back as its opposite, an antifluxon. Two superconductors joined across a thin barrier share their condensates' phases: a supercurrent tunnels across by the sine of the phase difference, and the phase difference turns at the rate of the voltage across, Josephson's two relations; across a wide junction it also changes with the magnetic field in the barrier, and with Maxwell's equations these make the sine-Gordon equation, whose waves travel at most at the Swihart velocity, as the Stockholm lecture notes on the [Josephson effect](https://ekmf.fysik.su.se/Supraledning/WWW/Lectures/Lecture9.pdf) and on [fluxons in Josephson junctions](https://ekmf.fysik.su.se/Supraledning/WWW/Lectures/Lecture12.pdf) set out.

The algebra is the junction's spacetime joined with a plane for the phase, `Algebra("x+y+t-") * Algebra("a+b+")`. The phase difference at each place is modelled as a direction in the phase plane, the bias direction `mv.a` turned by the phase, and its rate of change as a bivector in that plane, `rate * mv.ab`. Each direction is pulled towards `mv.a` by the tunnelling and towards its neighbours by the field's tension, and the torque on it is its wedge with the pull, `phases ^ pull`: the sine of the phase difference and the curvature of the phase in one product, with no angle unwrapped anywhere. The phase's gradient, `gradients ^ phases`, lies in the mixed planes `xa xb ya yb`, and its norm is the magnetic field's strength. The phase plane commutes with the junction's spacetime, so a boost, `(mv.xt * rapidity / 2).exp()`, or a turn of the junction's plane, `(mv.xy * angle / 2).exp()`, carries phase directions along unchanged: a moving fluxon is a resting one seen from a boosted frame, and an oblique one a straight one turned.

In the notation of the long junction, with lengths in Josephson penetration depths and times in inverse plasma frequencies, the phase difference obeys $\nabla^2\varphi - \varphi_{tt} = \sin\varphi$; a straight fluxon moving at $u$ is $\varphi = 4\arctan e^{\gamma(x - ut)}$ with $\gamma = 1/\sqrt{1 - u^2}$, contracted by $\sqrt{1 - u^2}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext
from numga.sparse import SparseExtensor, spdiag
from examples.electromagnetism.josephson import render, scenarios
from examples.electromagnetism.josephson.core import junction
from examples.util.animation import show_animation, show_states

# The junction's spacetime, and the plane of the phase, which no measure of length or time reaches.
ga = Algebra("x+y+t-") * Algebra("a+b+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Plane = ga.gatype.vector()                                                    # Vector+
Point = ga.gatype.antivector()                                                # Antivector-
Phase = Plane.derive.within(Algebra("a+b+"))                                  # Vector+
Rate = ga.gatype.bivector().derive.within(Algebra("a+b+"))                    # Bivector+
# A turn of the phase plane, which takes the bias direction to a phase.
Turn = (mv.ab * 0.0).exp().gatype                                             # Rotor+
# Unit steps along the junction's plane and a unit of time, as events from the origin.
axis = mv.x.dual()                                                            # [] Point-
across = mv.y.dual()                                                          # [] Point-
time = -mv.t.dual()                                                           # [] Point-

## The junction's dynamics

A junction is a grid of triangles over its plane, each place an event from the origin. The field's tension is the Laplacian of the phase directions: each hat's gradient read against the supercurrent another's drives along the junction, `gradients & supercurrent(gradients)`, with `supercurrent = Plane.dual()` in these units, taken over each place's share of the area. A fluxon at rest turns the phase once around across a few penetration depths, by four times the arctangent of the exponential of the distance along x; moving, it is the same kink read at each event from a boosted frame.

In [ ]:
# A gradient of the phase read as the supercurrent it drives along the junction, one in these units.
supercurrent = Plane.dual()                                                   # [] Point- <- Plane+


def operators(topology, places: np.ndarray) -> tuple[Point, Plane, SparseExtensor]:
    """A junction's places `Point-[places]`, its triangles' corners' hat gradients `[corners] Plane+[faces]`, and the
    field's tension `Scalar+[places <- places]`."""
    vertices = (axis * places[:, 0] + across * places[:, 1]).field()          # Point-[places]
    corners = topology.incidence[2, 0]                                        # [corners] Scalar+[faces <- places]
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)[..., None]  # [corners] Point-[faces]
    patches = (sides[..., 0] & sides[..., 1]) / 2                             # Antibivector-[faces]
    areas = patches.squared().square_root()                                   # Scalar+[faces]
    gradients = (sides | patches[..., None].inverse()) / 2                    # [corners] Plane+[faces]
    cells = (~corners * (areas / 3)).sum(axis=-1)                             # Scalar+[places]
    # Each hat's gradient read against the supercurrent another's drives, over the triangle.
    local = (gradients[..., :, None] & supercurrent(gradients[..., None, :])) * areas[..., None, None]   # [corners, corners] Scalar+[faces]
    stiffness = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=-1).sum(axis=-1)
    return vertices, gradients, spdiag(1 / cells) * stiffness


def fluxon(events: Point, rapidity: np.ndarray, sense: float) -> Turn:
    """The turns `[...] Turn+[places]` taking the bias direction to the phase of a fluxon line along y, or an
    antifluxon's for a negative sense, at these events `Point-[places]`, moving along x at each rapidity `[...]`."""
    # The resting frame's events: the events boosted back by the rapidity, read by the plane x.
    resting = mv.x & ((mv.xt * (-rapidity / 2)).exp() >> events)            # [...] Scalar+[places]
    # The kink turns by four times the arctangent of the exponential of that distance.
    angles = 4 * (sense * resting).exp().arctan()                             # [...] Scalar+[places]
    return (mv.ab * (-angles / 2)).exp()


def evolving(tension: SparseExtensor, phases: Phase, rates: Rate, dt: float, n_steps: int, stride: int
             ) -> Iterator[Phase]:
    """The phase directions `[...] Phase+[places]` every `stride` steps of `dt`, from these and these rates."""
    for step in range(n_steps + 1):
        if step % stride == 0:
            yield phases
        pull = mv.a - tension * phases                                        # [...] Phase+[places]
        rates = rates + dt * (phases ^ pull)                                  # [...] Rate+[places]
        phases = (rates * (-dt / 2)).exp() >> phases                          # [...] Phase+[places]


def started(turns_now: Turn, turns_next: Turn, dt: float) -> tuple[Phase, Rate]:
    """The phase directions of these turns now, and their rates: the wedge of the directions now and a step later,
    over the step."""
    now, later = turns_now >> mv.a, turns_next >> mv.a
    return now, (now ^ later) / dt

## A row of pendula

Along a narrow junction the phase differences are drawn as a row of pendula, the mechanical analogue of the sine-Gordon equation: each a pivot on the junction's axis with its tip displaced from it by the phase direction, hanging down where the junction rests. Gravity is the tunnelling, pulling each pendulum down, and a twisted spring joining them is the field's tension. A fluxon is one full turn of the pendula travelling along the row, an antifluxon the opposite turn; here they run into each other and pass through. The pendula are coloured by the tunnelling current, the sine of each one's angle, as the wide junctions below are. This cell takes about five seconds, most of it stepping the strip.

In [ ]:
# A strip so narrow that its fluxons stay straight, its places, and every so many places along its middle drawn.
strip, strip_places = junction(scenarios.n_along, scenarios.n_strip_rows, scenarios.strip_length, scenarios.strip_height)
strip_vertices, strip_gradients, strip_tension = operators(strip, strip_places)
dt = scenarios.strip_dt
middle_row = np.isclose(strip_places[:, 1], 0.0, atol=1e-9) & (np.abs(strip_places[:, 0]) <= scenarios.pendulum_reach)
shown = np.flatnonzero(middle_row)[::scenarios.pendulum_every]               # [pendula]
# The fluxon from the left and the antifluxon from the right, their turns composed so that their phases add.
meeting_events = [(axis * strip_places[:, 0] + across * strip_places[:, 1] + time * moment).field()
                  for moment in (0.0, dt)]                                    # [2] Point-[places]
pairs = [fluxon(events + axis * scenarios.meeting_start, scenarios.meeting_rapidity, 1.0) *
         fluxon(events - axis * scenarios.meeting_start, -scenarios.meeting_rapidity, -1.0) for events in meeting_events]
phases, rates = started(*pairs, dt)
meeting_frames = evolving(strip_tension, phases, rates, dt, scenarios.n_meeting, scenarios.meeting_stride)
show_animation(render.animate_pendula(strip_vertices.batch()[shown], (phases.batch()[shown] for phases in meeting_frames)),
               "josephson_pendula", scenarios.frame_duration_ms)

The figure shows the pendula on the junction's axis, seen obliquely. Each hangs down at rest and swings once over the top where a fluxon passes, red then blue for the fluxon and blue then red for the antifluxon. The two turns meet in the middle, overlap and come out the other side unchanged.

## Fluxons in motion

Straight fluxon lines start at several rapidities as one batch, on the same narrow strip. The field at a fluxon's centre grows as it contracts, its flux one quantum, so its width relative to a resting one's is the resting field over its field. This cell takes about six seconds, most of it stepping the strip.

In [ ]:
# The boosted fluxons on the strip, at the start and a step later.
rapidities = scenarios.rapidities                                             # [rapidities]
starting = [(axis * (strip_places[:, 0] - scenarios.start) + across * strip_places[:, 1] + time * moment).field()
            for moment in (0.0, dt)]                                          # [2] Point-[places]
phases, rates = started(*(fluxon(events, rapidities, 1.0) for events in starting), dt)
runs = list(evolving(strip_tension, phases, rates, dt, scenarios.n_running, scenarios.n_running))
# The field's squared strength in each triangle: the phase's gradient, in the mixed planes of place and phase.
strip_corners = strip.incidence[2, 0]                                         # [corners] Scalar+[faces <- places]
before, after = ((strip_gradients ^ (strip_corners * run[..., None])).sum(axis=-1).scalar_norm_squared()
                 for run in (runs[0], runs[-1]))                              # [rapidities] Scalar+[faces] each
render.draw_contraction(np.tanh(rapidities), before, after);

The circles are the fluxons' widths relative to a resting one's as they start and the squares after their run, against their speed in units of the Swihart velocity, and the line is the Lorentz contraction, `sqrt(1 - speed ** 2)`. The fluxons keep their contracted shape as they run.

## A ring of flux

A closed loop of fluxon line, a ring where the phase turns once around from inside to outside, has tension and no force to hold it: it collapses, accelerating towards the junction's light speed, passes through itself at the middle and comes out the other side as a ring of the opposite sense, shedding plasma waves. The colour is the tunnelling current, the bias direction wedged with the phase, `mv.a ^ phases`.

In [ ]:
square, square_places = junction(scenarios.n_square, scenarios.n_square, scenarios.square_side, scenarios.square_side)
square_vertices, _, square_tension = operators(square, square_places)
# The resting kink across the ring, by each place's distance from the middle less the ring's radius: four times the
# arctangent of its exponential.
distances = square_vertices.squared().abs().square_root()                     # Scalar+[places]
ring_angles = 4 * (distances - scenarios.ring_radius).exp().arctan()          # Scalar+[places]
ring_phases = (mv.ab * (-ring_angles / 2)).exp() >> mv.a                      # Phase+[places]
ring_run = evolving(square_tension, ring_phases, 0 * (mv.a ^ ring_phases), scenarios.ring_dt, scenarios.n_ring_steps,
                    scenarios.ring_stride)
# The tunnelling current at each frame, the bias direction wedged with the phase, drawn as it comes; the animation hands
# back the currents it drew.
ring_currents = show_states((mv.a ^ phases for phases in ring_run), render.animate, square, square_vertices,
                            name="josephson_ring", duration_ms=scenarios.frame_duration_ms)   # [frames] Rate+[places]

The colours show the tunnelling current, red where it runs one way across the barrier and blue where it runs the other; a fluxon line carries red on its inner side and blue on its outer. The ring shrinks, ever faster, vanishes at the middle and returns with its colours swapped, a ring of antifluxon, ringing out waves across the junction.

## Reflection at an edge

An oblique fluxon line, a straight one turned in the junction's plane, runs towards the junction's right edge. A free edge holds no field, so the line's ends straighten where they meet the top and bottom edges; at the right edge the line is reflected as an antifluxon, the junction's own mirror image of it, and runs back.

In [ ]:
wide, wide_places = junction(scenarios.n_wide, scenarios.n_high, scenarios.wide, scenarios.high)
wide_vertices, _, wide_tension = operators(wide, wide_places)
# The line's events: the junction's events, shifted to the line's start and turned by its tilt.
tilt = (mv.xy * (scenarios.tilt / 2)).exp()                                   # [] Rotor+
line_events = [tilt >> (axis * (wide_places[:, 0] - scenarios.line_start) + across * wide_places[:, 1] +
                        time * moment).field() for moment in (0.0, scenarios.line_dt)]   # [2] Point-[places]
phases, rates = started(*(fluxon(events, scenarios.line_rapidity, 1.0) for events in line_events), scenarios.line_dt)
line_run = evolving(wide_tension, phases, rates, scenarios.line_dt, scenarios.n_line_steps, scenarios.line_stride)
# The tunnelling current at each frame, drawn as it comes; the animation hands back the currents it drew.
line_currents = show_states((mv.a ^ phases for phases in line_run), render.animate, wide, wide_vertices,
                            name="josephson_reflection", duration_ms=scenarios.frame_duration_ms)   # [frames] Rate+[places]

The fluxon line, red on its left and blue on its right, runs into the right edge and comes back blue on its left and red on its right: an antifluxon, leaving waves behind it.

In [ ]:
# checks
# The fluxons' widths, the resting field over their field at the centre, follow the Lorentz contraction within three
# percent, as they start and after their run.
speeds = np.tanh(rapidities)
for strengths in (before, after):
    peaks = np.sqrt(strengths.to_array().max(axis=-1))
    np.testing.assert_allclose(peaks[0] / peaks, np.sqrt(1 - speeds ** 2), rtol=0.03)
# The ring collapses: halfway to its passing through the middle, the current's outermost strong place has moved in.
reach = [distances.to_array()[ring_currents[index].norm() > 0.5].max() for index in (0, 30)]
assert reach[1] < reach[0] - 2
# The fluxon comes back an antifluxon: along the junction's middle, its current runs one way to the left of the line
# and the other to its right before, and the reverse after.
middle = np.isclose(wide_places[:, 1], 0.0, atol=1e-9)
order = [np.sign(wide_places[middle, 0][np.argmax(current)] - wide_places[middle, 0][np.argmin(current)])
         for current in (line_currents[index].batch().kernel[middle, 0] for index in (0, -1))]
assert order[0] == -order[1]